<h1>🧬 ADSP — Step 1: variants that map to a protein-coding gene</h1>

**In one line:** for each variant, take the genes VEP annotated it against, and
keep the variant if at least one of those genes is protein-coding.

The variant's consequence does not matter at this step. An intron, UTR,
synonymous or upstream variant of a protein-coding gene is kept. Only variants
whose sole links are to lncRNAs, pseudogenes and similar are dropped. One row of
the product is a **(variant, protein-coding gene)** pair, which is what step 2
consumes.

## The method

> "You want to filter only variants that map to protein coding genes... using
> VEP, because VEP will differentiate between them." — Diane Xue, 2026-08-28
>
> "I think what you should do is drop the genes first, rather than dropping the
> variants. Restrict only to genes that are protein coding, because you have
> those labels." — Diane Xue

Three operations, in this order:

1. **Restrict the gene list** to genes whose HGNC locus group is
   `protein-coding gene`. This happens before any variant is looked at.
2. **Map each variant to genes through VEP**, and resolve the name VEP used to
   a BF4 gene entity.
3. **Keep the variant** if any of its genes survived step 1.

VEP says *which gene*; HGNC says *what kind of gene it is*. The two questions
are answered by different sources on purpose, which is why this takes two
reports — `annotate_variant` and `annotate_gene` — and not one.

### A variant does not have to be inside the gene

VEP annotates a variant against a gene when it falls within the gene **or**
within ~5 kb of either end — the promoter and downstream regions, which control
how much protein the gene makes. Two rows from the output:

| variant | gene | relation | why it is linked |
| --- | --- | --- | --- |
| `1:722408:G:C` | OR4F16 | upstream of gene | in the promoter region, before the gene starts |
| `1:702358:G:A` | OR4F16 | within gene | an intron variant across 13 transcripts of OR4F16 |

Neither changes the protein. Both can change how much of it is produced, which
is why they are kept.

## Results

Measured on bundle `9b8419b48be5004e`, the complete build. The cells below
recompute every number.

```
711,836   input variants
711,651   matched in the bundle                       (185 not in the reference)
──────────────────────────────────────────
356,268   VEP links to a protein-coding gene
    624   minus variants whose only gene has no HGNC locus type
355,644   DELIVERED, as 380,775 variant x gene rows        (49.97% of matched)
```

Why the rest were dropped:

```
179,630   no gene in the VEP annotation at all — intergenic
104,998   VEP links only to lncRNA / pseudogene
 71,379   VEP names a gene BF4 has no entity for
```

`intron_variant` is 79.6% of all delivered rows. Expected and wanted: being
inside a gene mostly means being inside an intron, and an intron variant can
change how much protein is made without changing the protein itself.

| relation to gene | rows |
| --- | ---: |
| within gene | 329,627 |
| downstream of gene | 26,361 |
| upstream of gene | 24,787 |

93.5% of the delivered variants touch exactly one gene, 6.1% touch two.

## Outputs

Written to `outputs/` by §9, or by the script in §10.

| file | rows | purpose |
| --- | ---: | --- |
| `step1_per_variant_coding.csv` | 380,775 | **the product.** One row per (variant, coding gene). This is what step 2 consumes. |
| `step1_per_variant.csv` | 711,836 | **the audit.** Every input, including dropped ones, with the reason. |
| `step1_summary.csv` | — | the counts above |

The product carries `consequence_category` (`coding` / `non_coding` /
`regulatory` / `intergenic`) because step 2 partitions on it, and this is
where the value already exists.

Join downstream on `gene_entity_id`, not on `coding_gene`: the displayed name is
the one VEP used, which may be an alias of the entity it resolved to. Those ids
are scoped to the bundle that produced them, which is why the product ships a
`.provenance.json` beside it.

## What step 2 does next

Narrows these by functional consequence, as a partition: coding variants get a
protein-altering filter, non-coding variants have to be a brain eQTL **and**
pQTL for the same gene. Step 1 asked *is there a protein-coding gene this
variant could affect?*; step 2 asks *what does it do to that gene?*

### 1. Open a bundle

In [1]:
import json
import time
from pathlib import Path

import pandas as pd

from biofilter import Biofilter

# No bundle argument: it comes from .biofilter.toml. On the LPC, point at the
# deployed one instead — a bundle is a directory, the one holding manifest.json:
#   bf = Biofilter(bundle="/project/hall_shared/datasets/biofilter/20260914")
bf = Biofilter(debug_mode=False)

# Results land here whatever directory the kernel was started in — VS Code and
# Jupyter disagree about that, and a bare filename ends up wherever they landed.
# The project root is the folder holding .biofilter.toml.
_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
ADSP = _root / "notebooks" / "Andre" / "adsp"
DATA_DIR = ADSP / "data"
OUTPUT_DIR = ADSP / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bf

[INFO] ════════════════════════════════════

[INFO] 🚀 Initializing Biofilter

[INFO]    • Version: 4.3.0

[INFO]    • Debug mode: False

[INFO]    • Config: /Users/andrerico/Works/Sys/biofilter_430/.biofilter.toml

...

[INFO]    • DB:     /Users/andrerico/Works/Sys/biofilter_430/biofilter_data/bundles/20260914/tables

[INFO]    • Views:  37 (read-only)

[INFO]    • Time:   280.0 ms

[INFO] ════════════════════════════════════

<Biofilter(db_uri=parquet:///Users/andrerico/Works/Sys/biofilter_430/biofilter_data/bundles/20260914)>

### 2. The input

One variant per line, `chr:pos:ref:alt`. The report parses these itself —
`chr` prefixes, `:` / `-` / `_` separators and `X` / `Y` / `MT` are all
accepted — so nothing here needs to normalise them.

Chromosome and position are parsed anyway, for one reason only: the report
returns them as null for an input it could not match, and the audit file in
§8 still has to place those rows.

In [2]:
CHROM_CODE = {"X": 23, "Y": 24, "M": 25, "MT": 25}

raw = pd.read_csv(DATA_DIR / "adsp_variants.csv", header=None, usecols=[0],
                  names=["variant"], dtype=str)
raw["variant"] = raw.variant.str.strip()
inputs = raw[raw.variant.notna() & raw.variant.ne("")].drop_duplicates()

parts = inputs.variant.str.extract(r"^(?:chr)?([0-9]{1,2}|[XYxyMm][Tt]?)[:_\-](\d+)", expand=True)
inputs = inputs.assign(
    chromosome=pd.to_numeric(parts[0].str.upper().replace(CHROM_CODE), errors="coerce"),
    position=pd.to_numeric(parts[1], errors="coerce"),
).reset_index(drop=True)

print(f"{len(inputs):,} input variants")
inputs.head()

711,836 input variants

,variant,chromosome,position
0,1:633963:C:T,1,633963
1,1:702358:G:A,1,702358
2,1:722408:G:C,1,722408
3,1:734153:G:C,1,734153
4,1:758443:G:C,1,758443


### 3. What VEP links each variant to — `annotate_variant`

gnomAD ships VEP annotations for each variant against each transcript, and each
annotation names a gene by symbol or by Ensembl id. `annotate_variant` returns
**one row per transcript**, so 258k variants come back as ~2.9M rows.

**The cost follows the chromosomes the input touches, not the number of
variants.** Measured on this bundle, after a warm-up call — the first query in a
process costs ~8 s on its own, and timing without discarding that measures the
warm-up:

| input | variants | rows | time |
| --- | ---: | ---: | ---: |
| chr22 only | 20 | 23 | 0.6 s |
| chr22 only | 12,643 | 150,722 | 0.9 s |
| chr1 **+** chr22 | 20 | 246 | 2.3 s |
| the whole list | 711,836 | 8,164,005 | 149.9 s |

Twenty variants spread over two chromosomes cost more than 12,643 on one. The
report reads the partition files the input names and skips the rest, so what
you pay for is the ground covered, not the questions asked.

**The optimisation that follows from this does not work here, and it is worth
saying why.** If cost follows the chromosomes touched, splitting the list into
one call per chromosome ought to help — and on a six-chromosome bundle it did,
17.8 s against 40.3 s. Over the full list it does not:

| | time |
| --- | ---: |
| one call, all 711,836 | 149.9 s |
| 22 calls, one per chromosome | 156.5 s |

Identical rows, slightly slower. Same rule, opposite conclusion: this list
touches every chromosome, so both routes cover the same ground and splitting
only adds the cost of 22 calls instead of one. The split pays when the input
touches **few** partitions, not as a general optimisation.

`most_severe_only` is deliberately left off. It keeps one row per *variant*,
and this step needs one row per *variant × gene*: a variant touching two genes
has a most-severe consequence in each.

In [3]:
started = time.perf_counter()
variant_result = bf.report.run("annotate_variant", input_data=inputs.variant.tolist())
links = variant_result.to_pandas()

print(f"{variant_result.num_rows:,} rows in {time.perf_counter() - started:.1f}s")
print(f"bundle: {variant_result.provenance['bundle_id']}")
print(links.status.value_counts().to_string())

[INFO] Report 'annotate_variant' produced 8,164,005 rows in 143.57s from bundle 9b8419b48be5004e.

8,164,005 rows in 147.1s

bundle: 9b8419b48be5004e

status
ok           8163820
not_found        185

#### An input that does not resolve

`status = not_found` here means **not in this bundle**, which is not the same
claim as "not in gnomAD". On the complete build only 185 of the 711,836 inputs
fail to match, so this is now a small number — but it is still the bundle
speaking about itself, not about gnomAD.

Note also the difference between the two things the report calls absence:

- **`status = not_found`** — no variant in the bundle for that input.
- **`status = ok` with a null `gene_symbol`** — the variant is there, VEP just
  did not attribute it to any gene. Intergenic. It is dropped by this step, but
  for a different reason, and §8 keeps the two apart.

In [4]:
found = links[links.status != "not_found"]
matched = found.input_value.nunique()

print(f"matched in this bundle : {matched:,} of {len(inputs):,}")
print(f"chromosomes present    : {sorted(int(c) for c in found.chromosome.dropna().unique())}")
print(f"rows with no VEP gene  : {int(found.gene_symbol.isna().sum()):,}")

links[links.status == "not_found"][["input_value", "status", "note"]].head(3)


matched in this bundle : 711,651 of 711,836

chromosomes present    : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]

rows with no VEP gene  : 846,820

,input_value,status,note
215237,10:39900253:C:T,not_found,No variant in this bundle for the given chr_po...
215511,10:41529825:A:T,not_found,No variant in this bundle for the given chr_po...
1077814,12:34833163:G:T,not_found,No variant in this bundle for the given chr_po...


### 4. Drop the genes first — `annotate_gene`

The gene names VEP used, resolved to BF4 entities and labelled with their HGNC
locus group. `gene_symbol` first, Ensembl `gene_id` as the fallback — the same
name the lookup is keyed on.

`annotate_gene` resolves through aliases, so a VEP symbol that is not the
entity's primary symbol still lands on the right gene. Its `status` says how:

| status | meaning |
| --- | --- |
| `ok` | the name is the gene's primary symbol |
| `partial` | matched through an alias, or to a gene the bundle names differently |
| `not_found` | no entity in the bundle carries this name |

**`include_variant_summary=False` is not a micro-optimisation here.** That
parameter populates `variant_count_in_gene_range`, which counts the variants
inside each gene's coordinates — a range join against the variant tables. This
step never reads that column, and switching it off takes these 20,347 genes
from **30.4 s to 0.2 s** with every locus group identical. Ask for it only when
the count is the thing you want.

In [5]:
found = found.assign(vep_gene=found.gene_symbol.fillna(found.gene_id))
vep_gene_names = found.vep_gene.dropna().unique().tolist()
print(f"{len(vep_gene_names):,} distinct gene names to resolve")

# include_variant_summary=False: this step never reads
# variant_count_in_gene_range, and computing it costs 30.4s of the 30.6s.
gene_result = bf.report.run(
    "annotate_gene", input_data=vep_gene_names, include_variant_summary=False
)
gene_kinds = gene_result.to_pandas()[
    ["input_value", "entity_id", "ensembl_id", "gene_symbol",
     "gene_locus_group", "gene_locus_type"]
].rename(columns={"input_value": "vep_gene", "gene_symbol": "resolved_gene_symbol",
                  "ensembl_id": "gene_ensembl_id"})

print(gene_result.to_pandas().status.value_counts().to_string())
print()
print(gene_kinds.gene_locus_group.value_counts(dropna=False).to_string())

58,526 distinct gene names to resolve

[INFO] Report 'annotate_gene' produced 58,526 rows in 0.38s from bundle 9b8419b48be5004e.

status
ok           34064
not_found    14912
partial       9550

gene_locus_group
protein-coding gene    17349
None                   14912
pseudogene              9248
ncRNA                   8074
non-coding RNA          7671
other                    591
pseudo                   536
snoRNA                   143
scRNA                      2

**`not_found` here is the gap worth watching.** These are gene references VEP
makes that BF4 has no entity for — almost all `LOC*` / `ENSG*` lncRNAs and
pseudogenes without HGNC symbols, which BF4 legitimately does not carry. They
are not silently lost: §8 records them per variant in `unresolved_vep_genes`.

### 5. The filter

Two masks. The first is the criterion; the second is a provenance decision.

**`gene_locus_group == 'protein-coding gene'`** — the criterion, from HGNC.

**`gene_locus_type` present and not `'unknown'`** — a small number of
protein-coding genes come from the NCBI DTP rather than HGNC. They are `LOC*` /
`ENSG*` identifiers with no HGNC name and no locus type, so their
protein-coding status is our ETL's reading of NCBI, not an HGNC assertion. They
are excluded by default so the gene list has a single provenance: **HGNC, locus
group protein-coding, with a locus type assigned.** Set
`ALLOW_UNLABELLED_GENES = True` to keep them, and the cell prints what that
costs.

`'unknown'` is HGNC's placeholder for a missing value, not a locus type.

In [6]:
ALLOW_UNLABELLED_GENES = False

annotated = found.merge(gene_kinds, on="vep_gene", how="left")

is_coding = annotated.gene_locus_group.eq("protein-coding gene")
is_labelled = annotated.gene_locus_type.notna() & annotated.gene_locus_type.ne("unknown")
keep = is_coding if ALLOW_UNLABELLED_GENES else (is_coding & is_labelled)

print(f"protein-coding genes in the resolved list : "
      f"{int(gene_kinds.gene_locus_group.eq('protein-coding gene').sum()):,}")
print(f"rows on a protein-coding gene             : {int(is_coding.sum()):,}")
print(f"variants on a protein-coding gene         : {annotated.loc[is_coding, 'input_value'].nunique():,}")
print(f"variants after the HGNC-label restriction : {annotated.loc[keep, 'input_value'].nunique():,}")
print(f"cost of the restriction                   : "
      f"{annotated.loc[is_coding, 'input_value'].nunique() - annotated.loc[keep, 'input_value'].nunique():,} variants")

protein-coding genes in the resolved list : 17,349

rows on a protein-coding gene             : 5,917,361

variants on a protein-coding gene         : 356,268

variants after the HGNC-label restriction : 355,644

cost of the restriction                   : 624 variants

### 6. The product — one row per (variant, coding gene)

The ~2.9M transcript rows collapse to one row per (variant, gene), keeping the
most severe consequence VEP reported for that pair. A variant touching two
genes gets two rows, so "upstream of GENE" has somewhere to live.

A row reads as a sentence: *variant `1:919049:G:A`, gene SAMD11, **upstream of
gene**, because `upstream_gene_variant`, and SAMD11 is a protein-coding gene.*

In [7]:
coding = (
    annotated[keep]
    .sort_values("severity_rank")
    .drop_duplicates(["input_value", "entity_id"])
    .copy()
)
coding["relation_to_gene"] = coding.consequence.map({
    "upstream_gene_variant": "upstream of gene",
    "downstream_gene_variant": "downstream of gene",
}).fillna("within gene")
coding["n_coding_genes"] = coding.groupby("input_value").input_value.transform("size")

product = coding.rename(columns={
    "input_value": "variant", "vep_gene": "coding_gene", "entity_id": "gene_entity_id",
})[[
    "variant", "chromosome", "position", "coding_gene",
    "gene_entity_id", "gene_ensembl_id",
    "relation_to_gene", "consequence", "consequence_category", "severity_rank",
    "gene_locus_group", "gene_locus_type", "n_coding_genes",
]].sort_values(["chromosome", "position", "coding_gene"]).reset_index(drop=True)

# The report returns these as float64 — nullable ints that went through
# pandas. Nothing here is null after the filter, and step 2 joins on them.
for column in ("chromosome", "position", "gene_entity_id", "severity_rank"):
    product[column] = product[column].astype("int64")

print(f"{len(product):,} rows over {product.variant.nunique():,} variants")
product.head(8)


380,775 rows over 355,644 variants

,variant,chromosome,position,coding_gene,gene_entity_id,gene_ensembl_id,relation_to_gene,consequence,consequence_category,severity_rank,gene_locus_group,gene_locus_type,n_coding_genes
0,1:702358:G:A,1,702358,OR4F16,18620,ENSG00000284662,within gene,intron_variant,non_coding,28,protein-coding gene,gene with protein product,1
1,1:722408:G:C,1,722408,OR4F16,18620,ENSG00000284662,upstream of gene,upstream_gene_variant,regulatory,32,protein-coding gene,gene with protein product,1
2,1:919049:G:A,1,919049,SAMD11,31010,ENSG00000187634,upstream of gene,upstream_gene_variant,regulatory,32,protein-coding gene,gene with protein product,1
3,1:919788:G:A,1,919788,SAMD11,31010,ENSG00000187634,upstream of gene,upstream_gene_variant,regulatory,32,protein-coding gene,gene with protein product,1
4,1:921797:T:C,1,921797,SAMD11,31010,ENSG00000187634,upstream of gene,upstream_gene_variant,regulatory,32,protein-coding gene,gene with protein product,1
5,1:922063:C:T,1,922063,SAMD11,31010,ENSG00000187634,upstream of gene,upstream_gene_variant,regulatory,32,protein-coding gene,gene with protein product,1
6,1:925474:T:C,1,925474,SAMD11,31010,ENSG00000187634,within gene,intron_variant,non_coding,28,protein-coding gene,gene with protein product,1
7,1:927009:A:G,1,927009,SAMD11,31010,ENSG00000187634,within gene,intron_variant,non_coding,28,protein-coding gene,gene with protein product,1


### 7. Reading the result

**A variant does not have to be inside the gene.** VEP annotates a variant
against a gene when it falls within the gene **or** within ~5 kb of either end —
the promoter and downstream regions, which control how much protein the gene
makes. Neither an upstream nor an intron variant changes the protein; both can
change how much of it is produced, which is why they are kept.

**`gene_locus_group` is constant** — always `protein-coding gene`. It is
carried on every row so the filter is stated inline rather than remembered.

**Null is not zero here.** `n_coding_genes` is a count and is never null; a
null `gene_locus_type` cannot appear at all, because the filter removed those
rows. The place null still means something is the audit file in §8, where a
null `vep_coding_genes` means *no coding gene*, while a null
`unresolved_vep_genes` means *nothing went unresolved* — good news, not missing
data.

In [8]:
print(product.relation_to_gene.value_counts().to_string())
print()
print("genes per variant:")
print(product.groupby("variant", sort=False).size().value_counts().sort_index().head(5).to_string())
print()
print("most common consequences:")
print((product.consequence.value_counts(normalize=True).head(6) * 100).round(1).to_string())

relation_to_gene
within gene           329627
downstream of gene     26361
upstream of gene       24787

genes per variant:

1    332406
2     21517
3      1555
4       160
5         6

most common consequences:

consequence
intron_variant                        79.6
downstream_gene_variant                6.9
upstream_gene_variant                  6.5
3_prime_UTR_variant                    2.9
non_coding_transcript_exon_variant     1.3
5_prime_UTR_variant                    1.1

### 8. The audit — every input, with the reason

The product says what survived. This says what happened to everything else, so
no decision is taken on trust. The four fates are mutually exclusive:

| fate | column that shows it |
| --- | --- |
| not in this bundle | `found_in_db = False` |
| in the bundle, VEP names no gene | `has_vep_gene = False` |
| VEP names only non-coding genes | `vep_all_genes` set, `vep_coding_genes` null |
| VEP names a gene BF4 cannot resolve | `unresolved_vep_genes` set |

**`unresolved_coding_hint`** reads two labels from two different sources: the
transcript is `protein_coding` (gnomAD / VEP) and the gene has no BF4 entity to
ask HGNC about. It does not assert that these genes code for proteins — it
records that **VEP suggests they do and we could not confirm it**. Typical case:
`ENSG00000267561`, which Ensembl annotates as protein-coding, HGNC has not
named, and BF4 carries neither. Dropped by a missing reference, not by biology.

In [9]:
unresolved = annotated.vep_gene.notna() & annotated.entity_id.isna()

def joined(mask):
    return (annotated[mask].groupby("input_value")
            .vep_gene.agg(lambda s: ";".join(sorted(set(s.dropna()))))
            .replace("", pd.NA))

audit = inputs.rename(columns={"variant": "input_variant"}).set_index("input_variant")
audit["found_in_db"] = audit.index.isin(found.input_value)
audit["has_vep_gene"] = audit.index.isin(annotated.loc[annotated.vep_gene.notna(), "input_value"])
audit["maps_to_coding_gene"] = audit.index.isin(coding.input_value)
audit["vep_coding_genes"] = joined(keep)
audit["vep_all_genes"] = joined(annotated.vep_gene.notna())
audit["unresolved_vep_genes"] = joined(unresolved)
audit["unresolved_coding_hint"] = audit.index.isin(
    annotated[unresolved & annotated.biotype.eq("protein_coding")].input_value
)
audit = audit.reset_index()

# Only a variant the bundle actually holds can be *dropped* by the filter. An
# input the bundle has never seen was never a candidate, and counting it here
# would let a partial bundle look like a strict filter.
dropped = audit[audit.found_in_db & ~audit.maps_to_coding_gene]

summary = pd.DataFrame([
    ("bundle_id", variant_result.provenance["bundle_id"]),
    ("input_variants", len(inputs)),
    ("matched_in_bundle", matched),
    ("not_in_bundle", len(inputs) - matched),
    ("variants_mapped_to_coding_gene", int(audit.maps_to_coding_gene.sum())),
    ("variants_mapped_to_coding_gene_pct",
     round(100.0 * audit.maps_to_coding_gene.sum() / matched, 2) if matched else 0.0),
    ("variant_gene_rows", len(product)),
    ("dropped_non_coding_gene_only",
     int((dropped.has_vep_gene & dropped.unresolved_vep_genes.isna()).sum())),
    ("dropped_with_unresolved_vep_gene",
     int((dropped.has_vep_gene & dropped.unresolved_vep_genes.notna()).sum())),
    ("dropped_no_vep_gene", int((~dropped.has_vep_gene).sum())),
    ("unresolved_but_coding_biotype", int(dropped.unresolved_coding_hint.sum())),
], columns=["metric", "value"])

summary

,metric,value
0,bundle_id,9b8419b48be5004e
1,input_variants,711836
2,matched_in_bundle,711651
3,not_in_bundle,185
4,variants_mapped_to_coding_gene,355644
5,variants_mapped_to_coding_gene_pct,49.97
6,variant_gene_rows,380775
7,dropped_non_coding_gene_only,104998
8,dropped_with_unresolved_vep_gene,71379
9,dropped_no_vep_gene,179630


### 9. Export

`gene_entity_id` is scoped to the bundle that produced it, so the product does
not travel without saying which bundle that was. Migrated reports leave a
`.provenance.json` beside anything `ReportResult.write()` writes; this file is
derived from two of them, so it gets the same sidecar written by hand.

In [10]:
coding_path = OUTPUT_DIR / "step1_per_variant_coding.csv"
product.to_csv(coding_path, index=False)

audit.sort_values(["chromosome", "position"]).to_csv(OUTPUT_DIR / "step1_per_variant.csv", index=False)
summary.to_csv(OUTPUT_DIR / "step1_summary.csv", index=False)

Path(f"{coding_path}.provenance.json").write_text(json.dumps({
    "step": "adsp_step_01_coding_gene_filter",
    "bundle_id": variant_result.provenance["bundle_id"],
    "reports": ["annotate_variant", "annotate_gene"],
    "filter": {
        "gene_locus_group": "protein-coding gene",
        "require_hgnc_locus_type": not ALLOW_UNLABELLED_GENES,
    },
    "rows": len(product),
    "variants": int(product.variant.nunique()),
}, indent=2))

for path in sorted(OUTPUT_DIR.glob("step1_*")):
    print(f"{path.name:<45} {path.stat().st_size / 1e6:8.1f} MB")

step1_per_variant.csv                             48.0 MB

step1_per_variant_coding.csv                      55.9 MB

step1_per_variant_coding.csv.provenance.json       0.0 MB

step1_summary.csv                                  0.0 MB

### 10. The same thing on the command line

The whole notebook, as one job. This is what the LPC runs — same two reports,
same join, same three files.

```bash
python notebooks/Andre/adsp/step_01_adsp_coding_gene_filter.py \
    --input  notebooks/Andre/adsp/data/adsp_variants.csv \
    --bundle /project/hall_shared/datasets/biofilter/20260914 \
    --out-summary outputs/step1_summary.csv \
    --out-detail  outputs/step1_per_variant.csv \
    --out-coding  outputs/step1_per_variant_coding.csv
```

Add `--allow-unlabelled-genes` for the `ALLOW_UNLABELLED_GENES = True` variant
of §5.

### 11. Quick QA

In [11]:
checks = {
    "every row is on a protein-coding gene":
        bool(product.gene_locus_group.eq("protein-coding gene").all()),
    "no (variant, gene) pair appears twice":
        not product.duplicated(["variant", "gene_entity_id"]).any(),
    "gene_entity_id never null":
        not product.gene_entity_id.isna().any(),
    "n_coding_genes matches the rows per variant":
        bool((product.groupby("variant").size() == product.groupby("variant").n_coding_genes.first()).all()),
    "audit covers every input exactly once":
        len(audit) == len(inputs) and not audit.input_variant.duplicated().any(),
    "kept + dropped + unmatched = input":
        int(audit.maps_to_coding_gene.sum()) + len(dropped) + (len(inputs) - matched) == len(inputs),
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

print(f"\nbundle: {variant_result.provenance['bundle_id']}")
product.dtypes.to_frame("dtype")

PASS  every row is on a protein-coding gene

PASS  no (variant, gene) pair appears twice

PASS  gene_entity_id never null

PASS  n_coding_genes matches the rows per variant

PASS  audit covers every input exactly once

PASS  kept + dropped + unmatched = input


bundle: 9b8419b48be5004e

,dtype
variant,object
chromosome,int64
position,int64
coding_gene,object
gene_entity_id,int64
gene_ensembl_id,object
relation_to_gene,object
consequence,object
consequence_category,object
severity_rank,int64


### 12. Known gaps

**71,379 variants were dropped because the gene VEP names has no BF4 entity.**
Almost all are lncRNAs and pseudogenes without HGNC symbols (`LOC*`, `ENSG*`),
which BF4 legitimately does not carry. **1,395 of them carry
`unresolved_coding_hint`** in the audit file — VEP's transcript biotype says
`protein_coding` and there is no entity to ask HGNC about. Dropped by a missing
reference, not by biology. Typical case: `ENSG00000267561`, which Ensembl
annotates as protein-coding, HGNC has not named, and BF4 carries neither.

**The ADSP list is autosomal.** The bundle carries chromosomes 1-22 plus X and
Y; the inputs reach chromosomes 1-22 only, which is a property of the list
rather than of the bundle.

**Transcript multiplicity is left unresolved here, on purpose.** There are 10.6
transcripts per (variant, gene) on average (median 6, measured on a random
8,000 of the delivered variants), and this step collapses them by taking the
most severe consequence for the pair. `canonical` is populated in this bundle —
88.8% of (variant, gene) pairs have a canonical transcript — so
`annotate_variant(canonical_only=True)` is available if a downstream step wants
one transcript per gene instead. Step 1 does not, because it only asks whether
*any* annotation hits a protein-coding gene, so no transcript has to be chosen.

Note that canonical would **not** resolve the 6.5% of variants that touch more
than one gene — each gene has its own canonical transcript. VEP's `--pick` is
the flag for that, and we do not have it.